# 03 — Firm-Year Sentiment Panel

**Goal:** Aggregate Revelio Labs review-level data into a firm-year sentiment panel
for the gvkey-mapped S&P 500 universe constructed in notebook 02.

**Input:**
- `data/sp500_universe_with_gvkey.parquet` (6,535 firm-year rows with gvkey)
- `revelio_sentiment.sentiment_individual_reviews` (43.5M review rows globally)
- `revelio.company_mapping` (rcid → gvkey bridge)

**Output:** `data/sentiment_panel.parquet`
- One row per (gvkey, year)
- Mean overall rating per firm-year, plus sub-scores
- Review count per firm-year (for inverse-variance weighting later)

**Aggregation approach:**
- Reviews filtered to the rcids matching our universe's gvkeys
- Each review's `review_date` determines the year of aggregation
- Within each (gvkey, year) cell: mean of `rating_overall` plus sub-scores
- `n_reviews` retained to document the precision of each firm-year estimate

**Manual rcid supplement (2026-06-10):** United Airlines (gvkey 010795) is attached
to rcid 822516 ("United Airlines, Inc.") by hand — Revelio's mapping leaves the
listed parent's gvkey NULL and assigns the reviews-bearing subsidiary rcid to the
subsidiary gvkey 010484. See the supplement cell for details. We do NOT hand-curate
any other unmapped rcids (e.g., Delphi-era "Delphi" pages for Aptiv): every other
firm uses Revelio's own gvkey mapping, so Aptiv's sentiment begins 2017-12 (its
mapped rcid's coverage) even though its returns/fundamentals cover the full window.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import wrds

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_PROCESSED = Path.home() / "thesis" / "data"

universe = pd.read_parquet(DATA_PROCESSED / "sp500_universe_with_gvkey.parquet")
print(f"Universe: {len(universe):,} firm-year rows")
print(f"Unique gvkeys: {universe['gvkey'].nunique()}")

# Extract the unique gvkeys we need to pull sentiment for
target_gvkeys = sorted(universe["gvkey"].unique())
print(f"Will pull Revelio reviews for {len(target_gvkeys)} firms")
print(f"First 5 gvkeys: {target_gvkeys[:5]}")

Universe: 6,535 firm-year rows
Unique gvkeys: 717
Will pull Revelio reviews for 717 firms
First 5 gvkeys: ['001045', '001075', '001078', '001161', '001177']


In [2]:
# Read the WRDS username from ~/.pgpass (4th field) so the notebook runs headlessly
WRDS_USERNAME = next(
    line.split(":")[3]
    for line in (Path.home() / ".pgpass").read_text().splitlines()
    if "wrds" in line
)
db = wrds.Connection(wrds_username=WRDS_USERNAME)

Loading library list...


Done


In [3]:
gvkey_list_sql = ",".join([f"'{g}'" for g in target_gvkeys])

rcid_mapping = db.raw_sql(f"""
    SELECT DISTINCT rcid, gvkey, ticker, company
    FROM   revelio.company_mapping
    WHERE  gvkey IN ({gvkey_list_sql})
      AND  rcid IS NOT NULL
""")

print(f"Mapping rows: {len(rcid_mapping)}")
print(f"Unique gvkeys with rcid: {rcid_mapping['gvkey'].nunique()}")
print(f"Unique rcids: {rcid_mapping['rcid'].nunique()}")
print()

# Coverage check: how many universe gvkeys are missing from Revelio entirely?
gvkeys_with_rcid = set(rcid_mapping["gvkey"])
gvkeys_missing = set(target_gvkeys) - gvkeys_with_rcid
print(f"Universe gvkeys: {len(target_gvkeys)}")
print(f"With Revelio rcid: {len(gvkeys_with_rcid)}")
print(f"Missing from Revelio: {len(gvkeys_missing)}")
print()

# Check for multi-rcid gvkeys (one firm with multiple Revelio entries)
multi_rcid = rcid_mapping.groupby("gvkey")["rcid"].nunique()
multi_rcid_count = (multi_rcid > 1).sum()
print(f"Gvkeys with multiple rcids: {multi_rcid_count}")
if multi_rcid_count > 0:
    print(f"Max rcids for one gvkey: {multi_rcid.max()}")
    print()
    print("Sample multi-rcid cases:")
    high_count_gvkeys = multi_rcid[multi_rcid > 1].sort_values(ascending=False).head(5).index
    print(rcid_mapping[rcid_mapping["gvkey"].isin(high_count_gvkeys)]
          .sort_values(["gvkey", "rcid"]))

Mapping rows: 707
Unique gvkeys with rcid: 707
Unique rcids: 707

Universe gvkeys: 717
With Revelio rcid: 707
Missing from Revelio: 10

Gvkeys with multiple rcids: 0


In [4]:
# Manual rcid supplement (2026-06-10): United Airlines.
#
# revelio.company_mapping carries the listed parent "United Airlines Holdings,
# Inc." (rcid 22214941, ticker UAL, cusip 910047109) WITHOUT a gvkey — and that
# rcid holds zero reviews. The 8k+ United Airlines reviews sit under the
# wholly-owned operating subsidiary "United Airlines, Inc." (rcid 822516), which
# Revelio maps to the subsidiary gvkey 010484 rather than the parent gvkey
# 010795 that the universe (and Compustat/CRSP) use. Attach rcid 822516 to
# 010795 so United's reviews enter the panel. gvkey 010484 is not in the
# universe, so this creates no double-counting.
manual_rcids = pd.DataFrame([
    {"rcid": 822516.0, "gvkey": "010795", "ticker": "UAL",
     "company": "United Airlines, Inc. (manual: reviews-bearing subsidiary rcid -> parent gvkey)"},
])

assert "010795" not in set(rcid_mapping["gvkey"]), "010795 already mapped — supplement obsolete"
assert 822516.0 not in set(rcid_mapping["rcid"]), "rcid 822516 already present under another gvkey"
rcid_mapping = pd.concat([rcid_mapping, manual_rcids], ignore_index=True)

print(f"Mapping rows after manual supplement: {len(rcid_mapping)}")
print(f"Unique gvkeys with rcid: {rcid_mapping['gvkey'].nunique()}")
gvkeys_missing = set(target_gvkeys) - set(rcid_mapping["gvkey"])
print(f"Universe gvkeys still missing from Revelio: {len(gvkeys_missing)}")
print(f"  {sorted(gvkeys_missing)}")

Mapping rows after manual supplement: 708
Unique gvkeys with rcid: 708
Universe gvkeys still missing from Revelio: 9
  ['003024', '006097', '008446', '011555', '012978', '015708', '023546', '045167', '045169']


In [5]:
rcid_list = rcid_mapping["rcid"].tolist()
rcid_list_sql = ",".join([str(int(r)) for r in rcid_list])  # rcids are numeric

reviews_query = f"""
    SELECT  rcid,
            review_date,
            rating_overall,
            rating_work_life_balance,
            rating_compensation_and_benefits,
            rating_career_opportunities,
            rating_culture_and_values,
            rating_senior_leadership,
            reviewer_employment_status
    FROM    revelio_sentiment.sentiment_individual_reviews
    WHERE   rcid IN ({rcid_list_sql})
      AND   review_date BETWEEN '2010-01-01' AND '2024-12-31'
      AND   rating_overall IS NOT NULL
"""

print("Running query — this may take 2-10 minutes...")
reviews = db.raw_sql(reviews_query)
print(f"Pulled {len(reviews):,} reviews")

Running query — this may take 2-10 minutes...


Pulled 4,299,282 reviews


In [6]:
# Add a year column for grouping
reviews["review_date"] = pd.to_datetime(reviews["review_date"])
reviews["year"] = reviews["review_date"].dt.year

print(f"Shape: {reviews.shape}")
print(f"Date range: {reviews['review_date'].min().date()} to {reviews['review_date'].max().date()}")
print(f"Unique rcids: {reviews['rcid'].nunique()}")
print()

print("Reviews per year:")
print(reviews.groupby("year").size())
print()

print("Rating distribution (rating_overall):")
print(reviews["rating_overall"].describe())
print()

print("Missing rates per column:")
print(reviews.isna().mean().sort_values(ascending=False).round(3))
print()

print("Sample rows:")
print(reviews.head())

Shape: (4299282, 10)
Date range: 2010-01-01 to 2024-12-31
Unique rcids: 664

Reviews per year:
year
2010     29458
2011     35386
2012     60913
2013     82212
2014    135933
2015    228244
2016    243972
2017    256867
2018    226885
2019    240875
2020    325118
2021    696030
2022    655990
2023    542468
2024    538931
dtype: int64

Rating distribution (rating_overall):
count    4299282.0
mean      3.645408
std       1.208272
min            1.0
25%            3.0
50%            4.0
75%            5.0
max            5.0
Name: rating_overall, dtype: Float64

Missing rates per column:


rating_culture_and_values           0.241
rating_senior_leadership            0.235
rating_work_life_balance            0.220
rating_compensation_and_benefits    0.217
rating_career_opportunities         0.213
reviewer_employment_status          0.097
rcid                                0.000
review_date                         0.000
rating_overall                      0.000
year                                0.000
dtype: float64

Sample rows:
    rcid review_date  rating_overall  rating_work_life_balance  rating_compensation_and_benefits  rating_career_opportunities  rating_culture_and_values  rating_senior_leadership  \
0  218.0  2023-03-31             5.0                      <NA>                              <NA>                         <NA>                       <NA>                      <NA>   
1  218.0  2021-05-21             3.0                      <NA>                              <NA>                         <NA>                       <NA>                      <NA>   
2  21

In [7]:
# What are the actual values in reviewer_employment_status?
print(reviews["reviewer_employment_status"].value_counts(dropna=False).head(20))

reviewer_employment_status
REGULAR        2870861
PART_TIME       613491
<NA>            415115
INTERN          184871
CONTRACT        156048
FREELANCE        31155
TEMPORARY        18846
SELF_EMPLOY       4344
SEASONAL          3140
RESERVE            471
APPRENTICE         457
PER_DIEM           376
TRAINEE             95
PHD                 11
UNKNOWN              1
Name: count, dtype: Int64


In [8]:
# Filter to REGULAR + NA employment status
keep_status = reviews["reviewer_employment_status"].isin(["REGULAR"]) | \
              reviews["reviewer_employment_status"].isna()
reviews_filtered = reviews[keep_status].copy()
print(f"Reviews retained after filter: {len(reviews_filtered):,} ({len(reviews_filtered)/len(reviews):.1%})")
print()

# Aggregate to (rcid, year): mean of each rating, plus count
agg_specs = {
    "rating_overall": "mean",
    "rating_work_life_balance": "mean",
    "rating_compensation_and_benefits": "mean",
    "rating_career_opportunities": "mean",
    "rating_culture_and_values": "mean",
    "rating_senior_leadership": "mean",
}

# Compute the means
sentiment_panel = (
    reviews_filtered
    .groupby(["rcid", "year"])
    .agg(agg_specs)
    .reset_index()
)

# Add count of reviews separately (independent of NaN in sub-ratings)
n_reviews = (
    reviews_filtered
    .groupby(["rcid", "year"])
    .size()
    .reset_index(name="n_reviews")
)

# Merge counts into the panel
sentiment_panel = sentiment_panel.merge(n_reviews, on=["rcid", "year"])

# Rename columns to be cleaner
sentiment_panel = sentiment_panel.rename(columns={
    "rating_overall": "sentiment_overall",
    "rating_work_life_balance": "sentiment_wlb",
    "rating_compensation_and_benefits": "sentiment_comp",
    "rating_career_opportunities": "sentiment_career",
    "rating_culture_and_values": "sentiment_culture",
    "rating_senior_leadership": "sentiment_leadership",
})

print(f"Firm-year rows before gvkey merge: {len(sentiment_panel):,}")
print(f"Unique rcids: {sentiment_panel['rcid'].nunique()}")
print(f"Year range: {sentiment_panel['year'].min()} to {sentiment_panel['year'].max()}")
print()

# Quick sanity check
print("Summary statistics:")
print(sentiment_panel[["sentiment_overall", "sentiment_wlb", "sentiment_comp", "n_reviews"]].describe())

Reviews retained after filter: 3,285,976 (76.4%)



Firm-year rows before gvkey merge: 9,162
Unique rcids: 664
Year range: 2010 to 2024

Summary statistics:
       sentiment_overall  sentiment_wlb  sentiment_comp     n_reviews
count             9162.0         9136.0          9136.0   9162.000000
mean            3.448945       3.366254        3.548806    358.652696
std             0.519722       0.514692        0.482216   1302.148113
min                  1.0            1.0             1.0      1.000000
25%             3.149819        3.06383        3.258549     25.000000
50%             3.489761       3.398791        3.559512     88.500000
75%             3.788405       3.688889        3.857143    274.000000
max                  5.0            5.0             5.0  38921.000000


In [9]:
# Bring gvkey in via the rcid_mapping table
sentiment_panel = sentiment_panel.merge(
    rcid_mapping[["rcid", "gvkey"]],
    on="rcid",
    how="left"
)

print(f"Rows before filter: {len(sentiment_panel):,}")
print(f"Rows with non-null gvkey: {sentiment_panel['gvkey'].notna().sum():,}")
print()

# All rows should have a gvkey at this point, since we only pulled reviews
# for our rcid_mapping firms. If any are missing, something's off.
if sentiment_panel["gvkey"].isna().any():
    print("WARNING: some rows have no gvkey — investigate")

# Filter to sample window (2012-2024 — we keep 2024 for completeness)
sentiment_panel = sentiment_panel[
    (sentiment_panel["year"] >= 2012) & (sentiment_panel["year"] <= 2024)
].copy()

# Restrict columns to what we'll save
final = sentiment_panel[[
    "gvkey", "year", "n_reviews",
    "sentiment_overall", "sentiment_wlb", "sentiment_comp",
    "sentiment_career", "sentiment_culture", "sentiment_leadership",
]].sort_values(["gvkey", "year"]).reset_index(drop=True)

print(f"Final firm-year sentiment panel: {len(final):,} rows")
print(f"Unique firms (gvkey): {final['gvkey'].nunique()}")
print(f"Year range: {final['year'].min()} to {final['year'].max()}")
print()

# Check for unexpected duplicates
dup_check = final.duplicated(subset=["gvkey", "year"]).sum()
print(f"Duplicate (gvkey, year) pairs: {dup_check}")
print()

print("Firms with sentiment data per year:")
print(final.groupby("year")["gvkey"].nunique())

Rows before filter: 9,162
Rows with non-null gvkey: 9,162

Final firm-year sentiment panel: 8,069 rows
Unique firms (gvkey): 664
Year range: 2012 to 2024

Duplicate (gvkey, year) pairs: 0

Firms with sentiment data per year:
year
2012    574
2013    586
2014    600
2015    614
2016    614
2017    618
2018    621
2019    629
2020    634
2021    645
2022    646
2023    643
2024    645
Name: gvkey, dtype: int64


In [10]:
output_path = DATA_PROCESSED / "sentiment_panel.parquet"
final.to_parquet(output_path, index=False)

# Read back to verify
check = pd.read_parquet(output_path)
print(f"Saved {len(check):,} rows to {output_path.name}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")
print()

# Sanity check: how many universe firm-years have a sentiment match?
universe = pd.read_parquet(DATA_PROCESSED / "sp500_universe_with_gvkey.parquet")
universe["gvkey"] = universe["gvkey"].astype(str)
final["gvkey"] = final["gvkey"].astype(str)

merged_check = universe.merge(final, on=["gvkey", "year"], how="left")
matched = merged_check["sentiment_overall"].notna().sum()
print(f"Universe firm-year rows: {len(merged_check):,}")
print(f"With sentiment match: {matched:,} ({matched/len(merged_check):.1%})")
print()

# Coverage by year
print("Universe coverage by year:")
coverage = merged_check.groupby("year").agg(
    universe_firms=("gvkey", "nunique"),
    with_sentiment=("sentiment_overall", lambda x: x.notna().sum()),
).assign(coverage_pct=lambda d: d["with_sentiment"] / d["universe_firms"])
print(coverage.round(3))

Saved 8,069 rows to sentiment_panel.parquet
File size: 315.3 KB

Universe firm-year rows: 6,535
With sentiment match: 5,926 (90.7%)

Universe coverage by year:
      universe_firms  with_sentiment  coverage_pct
year                                              
2012             497           412.0         0.829
2013             497           418.0         0.841
2014             496           426.0         0.859
2015             497           440.0         0.885
2016             500           449.0         0.898
2017             499           457.0         0.916
2018             500           463.0         0.926
2019             500           470.0          0.94
2020             500           474.0         0.948
2021             500           480.0          0.96
2022             500           480.0          0.96
2023             500           478.0         0.956
2024             500           479.0         0.958


## Weighting Decision (Pre-Committed)

The sentiment panel retains `n_reviews` per firm-year so that the weighting choice
can be made at the regression stage, not the data-construction stage. The
specifications below are pre-committed before any regression results are observed:

**Headline specifications:**
- **H1 (portfolio sort, asset pricing):** Equal-weighted portfolios within sentiment
  quintiles. Each firm-year contributes equally to its quintile's return.
- **H2a (long-short alpha):** Equal-weighted long-short portfolio, FF5 + MOM factor
  regression with Newey–West standard errors.
- **H2b (operating-performance panel regressions):** Unweighted firm-year observations.

**Robustness specifications (reported alongside headlines):**
- **Threshold restriction:** Sample restricted to firm-years with at least 30
  underlying reviews. Tests whether headline results depend on noisy early-period
  observations.
- **Precision weighting (H2b only):** Analytical weights of √n_reviews applied in
  the panel regression. Tests whether precision-weighting changes the inference.

**Rationale:** The headline specifications follow the asset-pricing literature
(Edmans, 2011; Green et al., 2019) and minimise researcher degrees of freedom.
The robustness specifications address the documented temporal imbalance in
review precision — median reviews per firm-year rises from ≈60 in 2012 to
≈1,400 in 2021 — without making it the primary specification.

**Pre-commitment date:** 14.05.2026. No regression results were observed
prior to this decision.